<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day04-lab.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 4 lab: a manual sequence alignment, by hand {.unnumbered}

This lab is done **on paper (or in your head), not in this notebook.**
Two real, short protein fragments, a real substitution matrix, and a
gap penalty are given below -- work out the full dynamic-programming
matrix and the traceback yourself, the same way the book page's own
worked example (`GCAT` vs `GAC`) and the teaching notebook's `VHLT` vs
`VLC` exercise did, but this time with a real (not flat-identity)
substitution matrix, and with nothing pre-computed for you.

This notebook exists only to **check** your by-hand work once you have
an answer, cell by cell -- it does not compute the matrix for you
first. Run the setup cell below to see the inputs, do the work with pen
and paper, then use the two check cells to see whether you got it
right. Once everything checks out, answer the Day 4 lab quiz on Canvas
using your own results.


In [ ]:
import numpy as np
from Bio.Align import substitution_matrices

# Two real protein fragments -- not the same ones used anywhere else on
# the Day 4 page or its teaching notebook:
#   seqA = "HFGKE", human beta-globin (HBB, UniProt P68871), residues 118-122
#   seqB = "HGQE",  human myoglobin  (MYG, UniProt P02144), residues 25-28
seqA, seqB = "HFGKE", "HGQE"
gap = -4.0

blosum62 = substitution_matrices.load("BLOSUM62")
residues = sorted(set(seqA) | set(seqB))

print(f"seqA = {seqA!r}  ({len(seqA)} residues, human beta-globin fragment)")
print(f"seqB = {seqB!r}  ({len(seqB)} residues, human myoglobin fragment)")
print(f"gap penalty = {gap} (flat, linear -- same recurrence as the book page and teaching notebook)")
print()
print("BLOSUM62, restricted to just the residues appearing above (real values, Bio.Align.substitution_matrices):")
print("     " + "  ".join(f"{r:>2}" for r in residues))
for r1 in residues:
    print(f"{r1:>2}  " + "  ".join(f"{int(blosum62[r1, r2]):>2}" for r2 in residues))
print()

m, n = len(seqA) + 1, len(seqB) + 1
S = np.full((m, n), np.nan)
for i in range(m):
    S[i, 0] = i * gap
for j in range(n):
    S[0, j] = j * gap

print("The recurrence (same as the book page and teaching notebook):")
print("  S[i,j] = max( S[i-1,j-1] + s(seqA[i-1], seqB[j-1]),   # diagonal: match/mismatch")
print("               S[i-1,j] + gap,                          # up: gap in seqB")
print("               S[i,j-1] + gap )                         # left: gap in seqA")
print()
print("Border filled in (mechanical -- i*gap / j*gap), interior left for you to work out by hand:")
print("Rows = '-' + seqA, columns = '-' + seqB:")
print("        " + "     ".join(["-"] + list(seqB)))
for i, row_label in enumerate(["-"] + list(seqA)):
    print(row_label, " ", S[i])


In [ ]:
# Fill in every interior cell with the value you worked out by hand.
# Keys are (row, col) using the same 1-indexed scheme as the printed
# matrix above (row 1 = seqA[0], col 1 = seqB[0], etc.) -- leave a cell
# as None if you haven't worked it out yet.
matrix_guesses = {
    (1, 1): None, (1, 2): None, (1, 3): None, (1, 4): None,
    (2, 1): None, (2, 2): None, (2, 3): None, (2, 4): None,
    (3, 1): None, (3, 2): None, (3, 3): None, (3, 4): None,
    (4, 1): None, (4, 2): None, (4, 3): None, (4, 4): None,
    (5, 1): None, (5, 2): None, (5, 3): None, (5, 4): None,
}

def _score(a, b):
    if a == '-' or b == '-':
        return gap
    return blosum62[a, b]

def _real_matrix():
    real = S.copy()
    for i in range(1, m):
        for j in range(1, n):
            diag = real[i-1, j-1] + _score(seqA[i-1], seqB[j-1])
            up = real[i-1, j] + gap
            left = real[i, j-1] + gap
            real[i, j] = max(diag, up, left)
    return real

_real = _real_matrix()

print("Checking your matrix against the real one, cell by cell:")
print()
all_correct = True
for (i, j), guess in matrix_guesses.items():
    real_val = _real[i, j]
    if guess is None:
        print(f"  S[{i},{j}]: not filled in yet")
        all_correct = False
    elif float(guess) == real_val:
        print(f"  S[{i},{j}]: correct! ({real_val:.0f})")
    else:
        print(f"  S[{i},{j}]: your guess {guess} does not match the real value {real_val:.0f}")
        all_correct = False

print()
print("Every cell correct -- move on to the traceback below." if all_correct
      else "Fill in every value above (or fix any mismatches) and re-run this cell.")


In [ ]:
# Now trace back from the bottom-right corner to the top-left, following
# the winning move at each cell, and write out the two final aligned
# strings (use '-' for a gap). Format only -- this example is NOT the
# real answer for this exercise, just shows the shape expected: e.g.
# your_alignment = ("PQR", "P-R") would mean no gap in seqA and one gap
# in seqB (at the middle position) for a 3-letter seqA aligned to a
# 2-letter seqB. Work out the real one yourself.
your_alignment = (None, None)

def _real_traceback():
    trace = np.zeros((m, n, 2))
    for i in range(1, m):
        trace[i, 0, :] = (-1, 0)
    for j in range(1, n):
        trace[0, j, :] = (0, -1)
    for i in range(1, m):
        for j in range(1, n):
            diag = _real[i-1, j-1] + _score(seqA[i-1], seqB[j-1])
            up = _real[i-1, j] + gap
            left = _real[i, j-1] + gap
            if diag >= max(up, left):
                trace[i, j, :] = (-1, -1)
            elif up >= left:
                trace[i, j, :] = (-1, 0)
            else:
                trace[i, j, :] = (0, -1)
    outA, outB = "", ""
    i, j = len(seqA), len(seqB)
    while i > 0 or j > 0:
        di, dj = trace[i, j]
        i, j = i + int(di), j + int(dj)
        outA = ("-" if di == 0 else seqA[i]) + outA
        outB = ("-" if dj == 0 else seqB[j]) + outB
    return outA, outB

real_outA, real_outB = _real_traceback()

guessA, guessB = your_alignment
if guessA is None or guessB is None:
    print("Not filled in yet.")
elif guessA == real_outA and guessB == real_outB:
    print("Correct! Your traceback matches the real optimal alignment.")
    print(guessA)
    print(guessB)
else:
    print("Not quite -- the real alignment is:")
    print(real_outA)
    print(real_outB)
    print("(yours was:)")
    print(guessA)
    print(guessB)


## Done

Once the matrix check and the traceback check both say "correct," you
have everything the Day 4 lab quiz on Canvas asks for: the full matrix,
the final score, and the final alignment.
